# **COMP 3610 - Assignment 4: MLOps & Model Deployment**

**Name:** Samuel Soman  
**ID:** 816039318 

### **Project Overview**

This project focuses on deploying a trained machine learning model as a complete prediction service. Using the NYC Yellow Taxi Trip Records dataset and a regression model developed in Assignment 2, the goal is to predict `tip_amount` based on trip-related features such as distance, time, and fare details.
The project includes several key stages of the machine learning deployment pipeline. First, MLflow is used to track experiments, compare model performance, and manage model versions through the Model Registry. Next, the selected best-performing regression model is exposed through a REST API built with FastAPI, allowing users to send input data and receive predicted tip amounts. Proper input validation and error handling are also included to make the service reliable.
To make the application portable and easy to run, the entire system is containerized using Docker and Docker Compose. This allows all components to be started with a single command and makes the service suitable for deployment on different environments or cloud platforms.
Overall, this project demonstrates an end-to-end machine learning deployment workflow, moving from a trained model to a containerized, API-accessible prediction service.

This notebook documents the end-to-end process of deploying a trained NYC Yellow Taxi tip-prediction model as a containerised REST API, covering:

| Part | Topic | Weight |
|------|-------|--------|
| 1 | Experiment Tracking with MLflow | 25 % |
| 2 | Model Serving with FastAPI | 35 % |
| 3 | Containerization with Docker | 20 % |
| 4 | Documentation & Code Quality | 10 % |

---

### <u>**0.0 Setup & Data Preparation**</u>

We begin by installing dependencies that are already listed in `requirements.txt`, downloading the NYC taxi dataset, and engineering the same features used in Assignment 2.

In [ ]:
# Install dependencies (safe to re-run)
%pip install -q -r requirements.txt

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"NumPy  : {np.__version__}")
print(f"Pandas : {pd.__version__}")

### <u>**0.1 Download the Dataset**</u>

We use the **Yellow Taxi Trip Data (January 2024)** parquet file.  
The file is downloaded only if it is not already present.

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

TAXI_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
TAXI_FILE = DATA_DIR / "yellow_tripdata_2024-01.parquet"

if not TAXI_FILE.exists():
    print("Downloading taxi trip data …")
    import urllib.request
    urllib.request.urlretrieve(TAXI_URL, TAXI_FILE)
    print(f"Saved to {TAXI_FILE}")
else:
    print(f"Data already present at {TAXI_FILE}")

df = pd.read_parquet(TAXI_FILE)
print(f"Raw dataset shape: {df.shape}")

### <u>**0.2 Data Cleaning & Feature Engineering**</u>

The same cleaning rules and feature derivations from Assignment 2 are applied here.  
Borough one-hot features are **omitted** to keep the deployment API simpler without meaningfully hurting Random Forest performance (borough importances < 0.002 each in Assignment 2).

In [ ]:
# ---------- cleaning ----------
df = df[df["payment_type"] == 1].copy()          # credit-card only
df = df[(df["fare_amount"] > 0) & (df["fare_amount"] < 200)]
df = df[(df["trip_distance"] > 0) & (df["trip_distance"] < 100)]
df = df[(df["tip_amount"] >= 0) & (df["tip_amount"] < 100)]
df = df[(df["passenger_count"] > 0) & (df["passenger_count"] <= 9)]

# ---------- temporal features ----------
df["pickup_hour"]        = df["tpep_pickup_datetime"].dt.hour
df["pickup_day_of_week"] = df["tpep_pickup_datetime"].dt.dayofweek

# ---------- trip duration ----------
df["trip_duration_minutes"] = (
    (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"])
    .dt.total_seconds() / 60
)
df = df[(df["trip_duration_minutes"] > 0) & (df["trip_duration_minutes"] < 180)]

# ---------- derived features ----------
df["trip_speed_mph"] = np.where(
    df["trip_duration_minutes"] > 0,
    df["trip_distance"] / (df["trip_duration_minutes"] / 60),
    0,
)
df["trip_speed_mph"] = df["trip_speed_mph"].clip(upper=80)

df["log_trip_distance"] = np.log1p(df["trip_distance"])

df["fare_per_mile"] = np.where(
    df["trip_distance"] > 0,
    df["fare_amount"] / df["trip_distance"],
    0,
)
df["fare_per_minute"] = np.where(
    df["trip_duration_minutes"] > 0,
    df["fare_amount"] / df["trip_duration_minutes"],
    0,
)

df["is_weekend"] = (df["pickup_day_of_week"] >= 5).astype(int)

for col in ["congestion_surcharge", "Airport_fee"]:
    df[col] = df[col].fillna(0)

In [ ]:
# ---------- feature matrix ----------
FEATURE_COLS = [
    "passenger_count", "trip_distance", "RatecodeID",
    "fare_amount", "extra", "mta_tax", "tolls_amount",
    "improvement_surcharge", "congestion_surcharge", "Airport_fee",
    "pickup_hour", "pickup_day_of_week",
    "trip_duration_minutes", "trip_speed_mph",
    "log_trip_distance", "fare_per_mile", "fare_per_minute",
    "is_weekend",
]

X = df[FEATURE_COLS].copy()
y = df["tip_amount"].copy()

# Fill residual NaN with median
X = X.fillna(X.median())

print(f"Features : {X.shape[1]}")
print(f"Samples  : {X.shape[0]:,}")


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f"Train : {X_train.shape[0]:,}  |  Test : {X_test.shape[0]:,}")

---
# **Part 1: Experiment Tracking with MLflow**
---

We set up a **local MLflow tracking server** backed by SQLite so that the Model Registry is available.  
Before running the cells below, **start the MLflow tracking UI** in a separate terminal:

```bash
mlflow ui --port 5000
```

Then open <http://localhost:5000> in a browser.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Point to the local tracking server
mlflow.set_tracking_uri("http://localhost:5000")

# Create (or reuse) the experiment
mlflow.set_experiment("taxi-tip-prediction")

print(f"MLflow version : {mlflow.__version__}")
print(f"Tracking URI   : {mlflow.get_tracking_uri()}")
print(f"Experiment     : taxi-tip-prediction")

In [ ]:
def log_regression_metrics(y_true, y_pred):
    """Compute and log MAE, RMSE, R² to the active MLflow run."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2   = r2_score(y_true, y_pred)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

### <u>**Model 1 - Random Forest Regressor (baseline)**</u>
Train a Random Forest with default-style hyperparameters and log everything to MLflow.

In [ ]:
with mlflow.start_run(run_name="rf-baseline"):
    # Parameters
    params = {
        "n_estimators": 100,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "random_state": RANDOM_STATE,
    }
    mlflow.log_params(params)

    # Train
    rf_baseline = RandomForestRegressor(**params, n_jobs=-1)
    rf_baseline.fit(X_train, y_train)

    # Evaluate & log metrics
    preds = rf_baseline.predict(X_test)
    metrics_baseline = log_regression_metrics(y_test, preds)

    # Tags
    mlflow.set_tag("model_type", "RandomForest")
    mlflow.set_tag("dataset_version", "yellow_tripdata_2024-01")
    mlflow.set_tag("author", "Samuel Soman")

    # Log model artifact
    mlflow.sklearn.log_model(rf_baseline, "model")

    baseline_run_id = mlflow.active_run().info.run_id

print(f"RF Baseline  — run_id: {baseline_run_id}")
for k, v in metrics_baseline.items():
    print(f"  {k}: {v:.4f}")

### <u>**Model 2 - Random Forest Regressor (tuned)</u>**

This model uses the best hyperparameters found in Assignment 2 via `RandomizedSearchCV`.

In [ ]:
with mlflow.start_run(run_name="rf-tuned"):
    # Best hyperparameters from Assignment 2 tuning
    params = {
        "n_estimators": 166,
        "max_depth": 10,
        "min_samples_split": 6,
        "min_samples_leaf": 2,
        "max_features": None,
        "random_state": RANDOM_STATE,
    }
    mlflow.log_params(params)

    # Train
    rf_tuned = RandomForestRegressor(**params, n_jobs=-1)
    rf_tuned.fit(X_train, y_train)

    # Evaluate & log metrics
    preds_tuned = rf_tuned.predict(X_test)
    metrics_tuned = log_regression_metrics(y_test, preds_tuned)

    # Tags
    mlflow.set_tag("model_type", "RandomForest")
    mlflow.set_tag("dataset_version", "yellow_tripdata_2024-01")
    mlflow.set_tag("author", "Samuel Soman")
    mlflow.set_tag("tuning", "RandomizedSearchCV_best")

    # Log model artifact
    mlflow.sklearn.log_model(rf_tuned, "model")

    tuned_run_id = mlflow.active_run().info.run_id

print(f"RF Tuned     — run_id: {tuned_run_id}")
for k, v in metrics_tuned.items():
    print(f"  {k}: {v:.4f}")

### <u>**Model 3 - Linear Regression</u>**

A simple linear baseline to compare against the tree-based models (following the pattern in the MLflow Quickstart guide).

In [ ]:
with mlflow.start_run(run_name="linear-regression"):
    mlflow.log_params({"model_type": "LinearRegression"})

    lr = LinearRegression()
    lr.fit(X_train, y_train)

    preds_lr = lr.predict(X_test)
    metrics_lr = log_regression_metrics(y_test, preds_lr)

    # Tags
    mlflow.set_tag("model_type", "LinearRegression")
    mlflow.set_tag("dataset_version", "yellow_tripdata_2024-01")
    mlflow.set_tag("author", "Samuel Soman")

    # Log model artifact
    mlflow.sklearn.log_model(lr, "model")

    lr_run_id = mlflow.active_run().info.run_id

print(f"Linear Reg   — run_id: {lr_run_id}")
for k, v in metrics_lr.items():
    print(f"  {k}: {v:.4f}")

### **MLflow UI Evidence**

> **Action:** open <http://localhost:5000> → navigate to the *taxi-tip-prediction* experiment → verify all three runs are visible with parameters, metrics, and tags.
>
> **Insert a screenshot of the MLflow UI here** (showing all three logged runs).

*(Screenshot placeholder - paste after running)*

---
### **Task 1.2 - Model Comparison & Registry**
---

### Programmatic comparison of logged runs

We also retrieve and compare runs via the MLflow API.

In [ ]:
experiment = mlflow.get_experiment_by_name("taxi-tip-prediction")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

comparison_cols = [
    "run_id", "tags.mlflow.runName",
    "metrics.mae", "metrics.rmse", "metrics.r2",
]
print("Side-by-side comparison of all logged runs:\n")

### **Side-by-side comparison screenshot**

> **Action:** in the MLflow UI, select both runs → click **Compare** → take a screenshot of the comparison view.
>
> **Insert screenshot here.**

*(Screenshot placeholder — paste after running)*

### <u>**Best Model Analysis**</u>

Three models were compared:

| Model | Description |
|-------|-------------|
| **RF Baseline** | Random Forest with default parameters |
| **RF Tuned** | Random Forest with hyperparameters from Assignment 2's `RandomizedSearchCV` |
| **Linear Regression** | Simple linear model as a non-tree baseline |

The **tuned Random Forest** outperforms the other two across all three metrics:

- **Lower MAE** - the tuned model's mean absolute error is smallest, meaning its average per-trip tip prediction is closest to the true value.
- **Lower RMSE** - fewer extreme mispredictions compared to the baseline RF and Linear Regression.
- **Higher R²** - a larger proportion of tip variance is explained, confirming best overall fit.

Linear Regression provides a useful lower bound - it captures the linear relationship between fare and tip but cannot model the non-linear interactions that the Random Forest captures (e.g., interaction between trip distance and time of day).

The improvement of the tuned RF over the baseline RF stems from the hyperparameter search conducted in Assignment 2, which found optimal tree depth and leaf constraints that reduce over-fitting on the training set while retaining predictive signal.

### <u>**Register the best model in the MLflow Model Registry**</u>

In [ ]:
MODEL_REGISTRY_NAME = "taxi-tip-regressor"

# Register the tuned model from its run
model_uri = f"runs:/{tuned_run_id}/model"
reg_result = mlflow.register_model(model_uri=model_uri, name=MODEL_REGISTRY_NAME)

print(f"Registered model : {reg_result.name}")
print(f"Version          : {reg_result.version}")

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Add a version description documenting its performance
client.update_model_version(
    name=MODEL_REGISTRY_NAME,
    version=reg_result.version,
    description=(
        f"Tuned Random Forest Regressor (n_estimators=166, max_depth=10). "
        f"MAE={metrics_tuned['MAE']:.4f}, RMSE={metrics_tuned['RMSE']:.4f}, "
        f"R²={metrics_tuned['R2']:.4f}. Trained on NYC yellow_tripdata_2024-01."
    ),
)
print("Version description updated.")

### <u>**Transition model to Production stage**</u>

Following the MLflow Quickstart guide, we promote the registered version to the **Production** stage so it can be loaded by stage name.

In [ ]:
# Transition the registered version to Production
client.transition_model_version_stage(
    name=MODEL_REGISTRY_NAME,
    version=reg_result.version,
    stage="Production",
)
print(f"Model '{MODEL_REGISTRY_NAME}' version {reg_result.version} → Production")

### <u>**Load model from registry & make a sample prediction**</u>

In [ ]:
# Load the Production model from the Model Registry
loaded_model = mlflow.sklearn.load_model(
    f"models:/{MODEL_REGISTRY_NAME}/Production"
)

# Use a single sample from the test set
sample = X_test.iloc[[0]]
actual  = y_test.iloc[0]
predicted = loaded_model.predict(sample)[0]

print(f"Sample prediction from registry model (Production stage):")
print(f"  Actual tip  : ${actual:.2f}")
print(f"  Predicted   : ${predicted:.2f}")

### <u>**Save model for API deployment**</u>

Export the best model as a `joblib` file so the FastAPI app can load it without a running MLflow server.

In [ ]:
import joblib

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / "rf_regressor.joblib"

joblib.dump(rf_tuned, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}  ({MODEL_PATH.stat().st_size / 1024:.0f} KB)")

---
# **Part 2: Model Serving with FastAPI** 
---

### <u>**Task 2.1 - API Design & Implementation**</u>

The full API lives in **`app.py`**.  Key design decisions:

1. **Model loaded once at startup** via FastAPI's `lifespan` context manager - never reloaded per-request.
2. **Pydantic `TripInput` model** validates **13 input fields** (5+ with constraints): `trip_distance` (gt=0), `passenger_count` (ge=1, le=9), `fare_amount` (ge=0), `pickup_hour` (ge=0, le=23), `pickup_day_of_week` (ge=0, le=6), `trip_duration_minutes` (ge=0), `RatecodeID` (ge=1, le=6), etc.
3. **Derived features** (`trip_speed_mph`, `log_trip_distance`, `fare_per_mile`, `fare_per_minute`, `is_weekend`) are computed server-side in `compute_features()` so the caller only provides raw trip attributes.
4. **Response** includes `tip_amount` (rounded to 2 d.p.), `model_version`, and a UUID `prediction_id`.

Below is the complete `app.py` source:

In [ ]:
# Display app.py source for review
print(open("app.py").read())

### **<u>Task 2.2 — Enhanced API Features</u>**

The following endpoints extend the base API (all implemented inside `app.py`):

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/predict/batch` | POST | Accepts a list of up to **100** trip records; enforced via `max_length=100` on the Pydantic model. |
| `/health` | GET | Returns API status, `model_loaded` flag, `model_version`, and `uptime_seconds`. |
| `/model/info` | GET | Returns metadata: model name, version, feature names, and training metrics (MAE, RMSE, R²). |
| *(global)* | — | A `@app.exception_handler(Exception)` catches unexpected errors and returns a structured `{"error": ..., "detail": ...}` JSON body with HTTP 500, **without** exposing internal stack traces. |

### <u>**Task 2.3 - API Testing</u>**

Automated tests are defined in **`test_app.py`** using `pytest` and FastAPI's `TestClient`.  
Below is the test source and execution output.

In [ ]:
# Display test_app.py source for review
print(open("test_app.py").read())

In [ ]:
# Run the test suite from the notebook
!pytest test_app.py -v

### Swagger UI Documentation

Start the server in a terminal with:

```bash
uvicorn app:app --reload --port 8000
```

Then visit <http://localhost:8000/docs> for the auto-generated Swagger UI.

> **Action:** take a screenshot of the Swagger UI at `/docs` and paste it here.

*(Screenshot placeholder — paste after running)*

---
# **Part 3: Containerization with Docker** 
---

### **<u>Task 3.1 - Dockerfile & Image Building</u>**

The `Dockerfile` uses `python:3.11-slim` and follows best practices:
- Copies `requirements.txt` first for layer caching.
- Installs dependencies with `--no-cache-dir` for a smaller image.
- Copies only `app.py` and `models/`.
- Exposes port 8000 and starts `uvicorn`.

A `.dockerignore` excludes data files, notebooks, `__pycache__/`, `.git/`, etc.

In [ ]:
# Show Dockerfile
print(open("Dockerfile").read())

In [ ]:
# Show .dockerignore
print(open(".dockerignore").read())

### <u>**Build the Docker image**</u>

In [ ]:
!docker build -t taxi-tip-api .

### <u>**Report image size**</u>

In [ ]:
!docker images taxi-tip-api

### <u>**Run the container and verify accessibility**</u>

In [ ]:
# Start container in background
!docker run -d --name taxi-api-test -p 8000:8000 taxi-tip-api

In [ ]:
import time, requests

# Wait for container to start
time.sleep(5)

# Test prediction from outside the container
payload = {
    "trip_distance": 3.5,
    "passenger_count": 1,
    "fare_amount": 15.0,
    "pickup_hour": 14,
    "pickup_day_of_week": 2,
    "trip_duration_minutes": 12.0,
}

resp = requests.post("http://localhost:8000/predict", json=payload)
print(f"Status : {resp.status_code}")
print(f"Body   : {resp.json()}")

In [ ]:
# Clean up standalone container
!docker stop taxi-api-test
!docker rm taxi-api-test

---
### **Task 3.2 - Docker Compose & Deployment Demo**

The `docker-compose.yml` defines two services:

| Service | Image | Port | Purpose |
|---------|-------|------|---------|
| `api` | Built from `Dockerfile` | 8000 | Prediction API |
| `mlflow` *(bonus)* | `python:3.11-slim` + mlflow | 5000 | MLflow tracking server |

Docker Compose creates a shared network so the API can reach MLflow at `http://mlflow:5000`.

In [ ]:
# Show docker-compose.yml
print(open("docker-compose.yml").read())

### <u>**Start services with `docker compose up`**</u>

In [ ]:
!docker compose up -d --build

In [ ]:
!docker compose ps

### <u>**Make at least 3 prediction requests to the containerised API</u>**

In [ ]:
import time, requests

# Wait for services to start
time.sleep(10)

BASE = "http://localhost:8000"

# --- Request 1: Health check ---
r1 = requests.get(f"{BASE}/health")
print("Request 1 — GET /health")
print(f"  Status: {r1.status_code}  Body: {r1.json()}\n")

# --- Request 2: Single prediction ---
trip_a = {
    "trip_distance": 2.1,
    "passenger_count": 2,
    "fare_amount": 10.5,
    "pickup_hour": 8,
    "pickup_day_of_week": 0,
    "trip_duration_minutes": 9.0,
}
r2 = requests.post(f"{BASE}/predict", json=trip_a)
print("Request 2 — POST /predict (morning commute)")
print(f"  Status: {r2.status_code}  Body: {r2.json()}\n")

# --- Request 3: Single prediction (different scenario) ---
trip_b = {
    "trip_distance": 18.0,
    "passenger_count": 1,
    "fare_amount": 52.0,
    "pickup_hour": 22,
    "pickup_day_of_week": 5,
    "trip_duration_minutes": 35.0,
    "tolls_amount": 6.55,
    "Airport_fee": 1.75,
}
r3 = requests.post(f"{BASE}/predict", json=trip_b)
print("Request 3 — POST /predict (airport ride, weekend evening)")
print(f"  Status: {r3.status_code}  Body: {r3.json()}\n")

# --- Request 4: Batch prediction ---
r4 = requests.post(f"{BASE}/predict/batch", json={"trips": [trip_a, trip_b]})
print("Request 4 — POST /predict/batch (2 trips)")
print(f"  Status: {r4.status_code}  Body: {r4.json()}")

### **<u>Shut down cleanly</u>**

In [ ]:
!docker compose down

### <u>**Container size report**</u>

In [ ]:
!docker images taxi-tip-api --format "table {{.Repository}}\t{{.Tag}}\t{{.Size}}"

### Configuration Notes

- **Docker Desktop** must be installed and running.
- The API image size is reported above; it is built from `python:3.11-slim` (~150 MB base) plus Python packages and the model artefact.
- Environment variables (`MODEL_PATH`, `MODEL_VERSION`) are set in `docker-compose.yml` so no hard-coded paths exist in the application.
- The MLflow service installs `mlflow==2.22.0` on first start; subsequent starts reuse the cached layer.
- To rebuild after code changes: `docker compose up --build`.

---
## AI Tools Used

| Tool | Purpose |
|------|---------|
| **GitHub Copilot** |  |
| **ChatGPT / Claude** |  |
